# MaxDiff batch analysis

Run this notebook top to bottom (**Kernel → Restart & Run All**) once per analysis batch.

1. Set the parameters in the next cell.
2. With `SOURCE = "firestore"` the notebook pulls a fresh batch of all responses of the study into `analysis/data/`. With `SOURCE = "file"` it reads an existing batch file (e.g. simulated data).
3. Results land in `analysis/output/<study>/`: scores CSV, charts PNG, quality report.

Responses hold item **ids only**. Wording is joined from the local items file just for labels; if that file is missing, charts show ids.
Method notes: `docs/analysis.md`.

In [ ]:
# ---- Parameters ---------------------------------------------------------
STUDY_ID        = "restaurant-demo"
ITEMS_FILE      = "../design/studies/restaurant-demo.items.json"   # local, confidential wording
SOURCE          = "file"            # "file" or "firestore"
RESPONSES_FILE  = "data/restaurant-demo.simulated.jsonl"           # used when SOURCE == "file"
CREDENTIALS     = "~/keys/maxdiff-service-account.json"            # used when SOURCE == "firestore"
EXCLUDE_FLAGGED = True              # drop speeders and straight-liners
SEGMENT_QUESTION = "S1"             # or None
OUTPUT_DIR      = f"output/{STUDY_ID}"


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import models as m

pd.set_option("display.precision", 3)
out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)

if SOURCE == "firestore":
    import export
    RESPONSES_FILE = export.pull_batch(STUDY_ID, credentials=CREDENTIALS)

labels = m.load_items(ITEMS_FILE)
responses = [r for r in m.load_responses(RESPONSES_FILE) if r.get("studyId") == STUDY_ID]
print(f"Batch file: {RESPONSES_FILE}")
print(f"Responses for {STUDY_ID}: {len(responses)}")
print("Wording file:", "found" if labels["items"] else "NOT found (labels show ids only)")

## 1. Data quality

In [ ]:
q = m.quality_flags(responses)
summary = pd.Series({
    "completes": len(q),
    "median duration (s)": q["duration_sec"].median(),
    "median ms per screen": q["median_ms"].median(),
    "speeders": int(q["speeder"].sum()),
    "straight-liners": int(q["straightliner"].sum()),
    "flagged (either)": int(q["flagged"].sum()),
})
q.to_csv(out / "quality.csv", index=False)
display(summary.to_frame("value"))
display(q[q["flagged"]])

In [ ]:
keep = set(q.loc[~q["flagged"], "respondent"]) if EXCLUDE_FLAGGED else set(q["respondent"])
used = [r for r in responses if r["_id"] in keep]
long = m.to_long(used)
print(f"Respondents used: {len(used)} of {len(responses)}; long table rows: {len(long)}")
long.head()

## 2. Level 1: count scores

Score = (times best − times worst) ÷ times shown, from −1 to +1. The 95% interval resamples respondents.

In [ ]:
counts = m.count_scores(long)
counts["label"] = [m.label(i, labels) for i in counts["item"]]
counts.to_csv(out / "scores_counts.csv", index=False)
m.plot_ranked(counts, "score", "ci_low", "ci_high", labels,
              title=f"{STUDY_ID}: best − worst score (95% CI)", xlabel="score (−1 … +1)")
plt.savefig(out / "chart_counts.png", dpi=160, bbox_inches="tight"); plt.show()
counts

## 3. Level 2: logit utilities and share of preference (default for reporting)

Sequential best-worst logit, fitted by maximum likelihood. Share of preference sums to 100: the chance an item would be picked as best from the full list.

In [ ]:
fit = m.fit_mnl(long)
shares = m.preference_shares(fit)
shares["label"] = [m.label(i, labels) for i in shares["item"]]
shares.to_csv(out / "scores_logit.csv", index=False)
r2 = 1 - fit["loglik"] / fit["loglik_null"]
print(f"converged: {fit['converged']}   tasks: {fit['tasks']}   respondents: {fit['respondents']}")
print(f"log-likelihood {fit['loglik']:.1f} vs. random {fit['loglik_null']:.1f}  (McFadden R² {r2:.2f})")
print(f"reference item (utility 0): {fit['reference']}")
m.plot_ranked(shares, "share", "share_low", "share_high", labels,
              title=f"{STUDY_ID}: share of preference (95% CI)", xlabel="share of preference (%)")
plt.savefig(out / "chart_shares.png", dpi=160, bbox_inches="tight"); plt.show()
shares

## 4. Segments

In [ ]:
if SEGMENT_QUESTION:
    seg, n_by_opt = m.segment_scores(long, SEGMENT_QUESTION)
    opt_names = labels["segments"].get(SEGMENT_QUESTION, {}).get("options", {})
    seg.columns = [f"{c} {opt_names.get(c, '')} (n={n_by_opt[c]})".strip() for c in seg.columns]
    seg = seg.loc[shares["item"]]
    seg.index = [m.label(i, labels) for i in seg.index]
    seg.to_csv(out / f"scores_segment_{SEGMENT_QUESTION}.csv")
    ax = seg.iloc[::-1].plot.barh(figsize=(8, 0.45 * len(seg) + 1.2), width=0.8)
    q_text = labels["segments"].get(SEGMENT_QUESTION, {}).get("text", SEGMENT_QUESTION)
    ax.set_title(f"Share of preference by: {q_text}", loc="left"); ax.set_xlabel("share (%)")
    h, l = ax.get_legend_handles_labels(); ax.legend(h[::-1], l[::-1], loc="lower right")
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout()
    plt.savefig(out / f"chart_segment_{SEGMENT_QUESTION}.png", dpi=160, bbox_inches="tight"); plt.show()
    display(seg)

## 5. Validation (simulated data only)

If the batch came from `tools/simulate_respondents.py`, compare the estimates with the true utilities the simulator used. Rank correlation near 1 means the pipeline works.

In [ ]:
truth_path = Path(str(RESPONSES_FILE)).with_suffix(".truth.json")
if truth_path.exists():
    from scipy.stats import spearmanr
    tu = json.loads(truth_path.read_text())["trueUtilities"]
    est = fit["utilities"]
    rho = spearmanr([tu[i] for i in est.index], est.values).statistic
    rho_c = spearmanr([tu[i] for i in counts["item"]], counts["score"]).statistic
    print(f"Rank correlation with truth: logit {rho:.3f}, counts {rho_c:.3f}")
    comp = pd.DataFrame({"true": pd.Series(tu), "estimated": est}).sort_values("true", ascending=False)
    display(comp)
else:
    print("No truth file: real data, nothing to validate against.")

In [ ]:
print("Outputs written to", out.resolve())
for p in sorted(out.iterdir()): print(" ", p.name)